In [2]:
import pandas as pd
#from google.cloud import bigquery
from common_lib.sql import BigQueryConnector
import plotly.express as px
from common_lib.metrics import aggregate_metric, aggregate_segment_metric, aggregate_ratio_metric, aggregate_segment_ratio_metric, plot_metric

## AB test assign

In [ ]:
query_location = './sql/abtest_assign.sql'
parameters = {
    'experiment_name': 'Temp_Generator',
    'assignmentstartdate': '2026-08-05'
}

bqc = BigQueryConnector()
cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

This query will process 11.9 MB when run.
Estimated query cost: $0.00


In [4]:
refresh_data = True

In [5]:
data = pd.DataFrame()

if refresh_data:
    data = bqc.get(query='./sql/abtest_assign.sql', is_path=True, query_parameters=parameters)
    data.to_pickle('./data/abtest_assign_data.pkl')
else:
    data = pd.read_pickle('./data/abtest_assign_data.pkl')

In [6]:
data

,user_id,experiment_name,exposure_dt,assigned_dt,min_start_dt,end_dt,variant
0,7B23E10FD9D7B1EC,Temp_Generator,2026-08-17,2026-08-17,2026-08-05,2026-09-02,Test
1,AB37A973E0292A73,Temp_Generator,2026-08-17,2026-08-17,2026-08-05,2026-09-02,Test
2,AA0E75A9CFC9C5EC,Temp_Generator,2026-08-17,2026-08-17,2026-08-05,2026-09-02,Test
3,8E1EF80DA38D8CFB,Temp_Generator,2026-08-17,2026-08-17,2026-08-05,2026-09-02,Test
4,BC168FF7864907D9,Temp_Generator,2026-08-17,2026-08-17,2026-08-05,2026-09-02,Test
...,...,...,...,...,...,...,...
120826,3823DBF8F26D2A82,Temp_Generator,2026-08-23,2026-08-23,2026-08-05,2026-09-02,control
120827,7F49C69DBF8C14D,Temp_Generator,2026-08-23,2026-08-23,2026-08-05,2026-09-02,control
120828,EF254655C929210E,Temp_Generator,2026-08-23,2026-08-23,2026-08-05,2026-09-02,control
120829,EF78E401EA66CE59,Temp_Generator,2026-08-23,2026-08-23,2026-08-05,2026-09-02,control


## ALL metrics

In [7]:
query_location = './sql/abtest_metrics.sql'
parameters = {
    'lookback':28,
    'assignmentstartdate': '2026-08-05'
}

bqc = BigQueryConnector()
cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

This query will process 3.35 GB when run.
Estimated query cost: $0.02


In [8]:
if refresh_data:
    data_metrics = bqc.get(query='./sql/abtest_metrics.sql', is_path=True, query_parameters=parameters)
    data_metrics.to_pickle('./data/abtest_metrics.pkl')
else:
    data_metrics = pd.read_pickle('./data/abtest_metrics.pkl')

In [9]:
data_metrics

,user_id,install_dt,dt,days_since_install,max_level,max_gameday,active,active_cumu,days_since_last_active_exc_today,n_sessions,...,n_cashdash_ms_completed,n_datedash_ms_completed,n_space_ms_completed,n_art_ms_completed,n_roadtrip_ms_completed,n_fortunes_ms_completed,n_timedalbum_sets_completed,reached_600_clash_points,reached_rank1,loading_timestamp
0,A68C9CC717C71F93,2025-03-06,2026-08-19,531,151,198,1,532,1,6,...,0,2,0,0,0,0,0,0,0,2026-08-28 02:52:08.359994+00:00
1,2C40C3B597C0C56D,2025-03-11,2026-08-19,526,137,176,1,302,1,2,...,0,2,0,0,0,0,0,1,0,2026-08-28 02:52:08.359994+00:00
2,66ED2FD8ACEBDE71,2022-01-16,2026-08-19,1676,158,207,1,909,1,7,...,0,3,0,0,0,0,0,1,0,2026-08-28 02:52:08.359994+00:00
3,9B81C84AE5E46F98,2022-12-06,2026-08-19,1352,158,207,1,1312,1,7,...,0,2,0,0,0,0,0,1,0,2026-08-28 02:52:08.359994+00:00
4,7C3060B573C085BE,2024-03-02,2026-08-19,900,158,207,1,775,1,4,...,0,2,0,0,0,0,0,1,0,2026-08-28 02:52:08.359994+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3803506,D48CF3AF0F18E5FD,2025-12-17,2026-07-10,205,155,203,1,195,1,1,...,0,0,0,0,0,0,0,0,0,2026-07-15 03:14:16.149203+00:00
3803507,D0BDDCF23B617FED,2023-03-08,2026-07-10,1220,152,198,1,899,1,1,...,0,0,0,0,0,0,0,0,0,2026-07-15 03:14:16.149203+00:00
3803508,E80D9C4202D237D9,2025-07-16,2026-07-10,359,21,16,1,46,29,1,...,0,0,0,0,0,0,0,0,0,2026-07-15 03:14:16.149203+00:00
3803509,C7466E101D12948F,2026-07-04,2026-07-10,6,1,1,1,3,5,1,...,0,0,0,0,0,0,0,0,0,2026-07-15 03:14:16.149203+00:00


## Merge metrics and assignment

In [10]:
data_metrics = data_metrics.merge(data[['user_id','variant','assigned_dt']], on=['user_id'], how='left')
data_metrics['variant'] = data_metrics['variant'].fillna('NotAssigned')

# A row where dt < assigned_dt is a user's activity from before they were actually randomized
# into a variant - attributing it to Test/control would leak pre-assignment behavior into the
# comparison (assignment can start well before dt, since users get assigned continuously
# throughout the test, not just on day 1). Route those rows back to NotAssigned so
# aggregate_metric drops them the same way it already drops genuinely-unassigned users.
data_metrics.loc[data_metrics['dt'] < data_metrics['assigned_dt'], 'variant'] = 'NotAssigned'

### IAP Rev

In [11]:
metric = 'iap_rev'
data_metrics_agg = aggregate_metric(data_metrics, metric, 'mean', data)

plot_metric(
    data_metrics_agg, 
    metric,
    show_overall_diff=True, 
    show_value=True,
    show_rel=True, 
    show_value_cumulative=True,
    show_rel_cumulative=True,
    assignment_start_date='2026-08-05',
    test_start_date='2026-08-17',
    width=1500,
    height=400
)

## Static AB Test Metrics (production snapshot)

Same metrics, but read from `ab_dt_segment_metrics` - the pre-aggregated table [ab_trends.ipynb](../ab_tests_user_cuped_method/ab_trends.ipynb) itself reads from in production, rebuilt daily. Unlike everything above (which always reflects live data, late-arriving events included, up to the moment it's run), this is a static snapshot as of that table's last scheduled refresh (`loading_timestamp`). Comparing the two shows how much a given day's numbers have moved since the last refresh - useful for sanity-checking "why does the dashboard show a different number than my notebook" without re-deriving the whole chain by hand.

In [12]:
query_location = './sql/ab_dt_segment_metrics.sql'
parameters = {
    'experiment_name': 'Temp_Generator',
}

bqc = BigQueryConnector()
cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

This query will process 102.3 MB when run.
Estimated query cost: $0.00


In [13]:
data_segment_metrics = pd.DataFrame()

if refresh_data:
    data_segment_metrics = bqc.get(query=query_location, is_path=True, query_parameters=parameters)
    data_segment_metrics.to_pickle('./data/ab_dt_segment_metrics.pkl')
else:
    data_segment_metrics = pd.read_pickle('./data/ab_dt_segment_metrics.pkl')

In [14]:
data_segment_metrics

,dt,variant,metric,total_users_assigned,dau_assigned,dau,sum_metric,avg_active_metric,sd_active_metric,loading_timestamp
0,2026-08-09,control,last_n_tiles_free,30651,30651.0,30651.0,360358.000000,11.756811,6.890624,2026-08-25 15:53:35.654456+00:00
1,2026-08-09,control,first_n_tiles_free,30651,30651.0,30651.0,366443.000000,11.955336,6.786158,2026-08-25 15:53:35.654456+00:00
2,2026-08-09,control,min_n_tiles_free,30651,30651.0,30651.0,75318.000000,2.457277,4.007460,2026-08-25 15:53:35.654456+00:00
3,2026-08-09,control,avg_n_tiles_free,30651,30651.0,30651.0,323858.657871,10.566006,5.508009,2026-08-25 15:53:35.654456+00:00
4,2026-08-09,control,max_n_tiles_free,30651,30651.0,30651.0,550606.000000,17.963721,6.533177,2026-08-25 15:53:35.654456+00:00
...,...,...,...,...,...,...,...,...,...,...
8535,2026-08-06,control,n_seasonalevent_ms_completed,2174,2153.0,2153.0,0.000000,0.000000,0.000000,2026-08-25 15:53:35.654456+00:00
8536,2026-08-06,control,n_trans_sale,2174,2153.0,2153.0,0.000000,0.000000,0.000000,2026-08-25 15:53:35.654456+00:00
8537,2026-08-06,control,n_trans_lops,2174,2153.0,2153.0,0.000000,0.000000,0.000000,2026-08-25 15:53:35.654456+00:00
8538,2026-08-06,control,net_iap_rev_lops,2174,2153.0,2153.0,0.000000,0.000000,0.000000,2026-08-25 15:53:35.654456+00:00


In [15]:
#data_segment_metrics.metric.unique()

In [16]:
metric = 'iap_rev'
data_segment_metrics_agg = aggregate_segment_metric(data_segment_metrics, metric)

plot_metric(
    data_segment_metrics_agg,
    metric,
    show_overall_diff=True,
    show_value=True,
    show_rel=True,
    show_value_cumulative=True,
    show_rel_cumulative=True,
    assignment_start_date='2026-08-05',
    test_start_date='2026-08-17',
    width=1500,
    height=400
)

### Energy

In [17]:
metric_name = 'energy_spent'

data_energy_velocity = aggregate_segment_metric(
    data_segment_metrics, 
    metric_name=metric_name)

plot_metric(
    data_energy_velocity,
    metric_name,
    show_overall_diff=False,
    show_value_cumulative=False,
    show_rel_cumulative=False,
    test_start_date='2026-08-17',
    assignment_start_date='2026-08-05',
    width=1500,
    height=400
)

metric_name = 'energy_earned'

data_energy_velocity = aggregate_segment_metric(
    data_segment_metrics, 
    metric_name=metric_name)

plot_metric(
    data_energy_velocity,
    metric_name,
    show_overall_diff=False,
    show_value_cumulative=False,
    show_rel_cumulative=False,
    test_start_date='2026-08-17',
    assignment_start_date='2026-08-05',
    width=1500,
    height=400
)

data_energy_velocity = aggregate_segment_ratio_metric(
    data_segment_metrics, 
    numerator='energy_spent', 
    denominator='energy_earned', 
    ratio_name='energy_velocity')

# Ratio metrics have no cumulative form here (nor in production - energy_velocity etc. only ever
# get "Active" charts in ab_trends.ipynb's own output, never "Cumulative" ones), so the
# cumulative/overall-diff pieces are turned off rather than left to fail on the missing
# sum_energy_velocity/total_users_assigned columns those need.
plot_metric(
    data_energy_velocity,
    'energy_velocity',
    show_overall_diff=False,
    show_value_cumulative=False,
    show_rel_cumulative=False,
    test_start_date='2026-08-17',
    assignment_start_date='2026-08-05',
    width=1500,
    height=400
)

### Gems

In [20]:
metric_name = 'gems_spent'

data_gems_spent = aggregate_segment_metric(
    data_segment_metrics, 
    metric_name=metric_name)

plot_metric(
    data_gems_spent,
    metric_name,
    show_overall_diff=False,
    show_value_cumulative=False,
    show_rel_cumulative=False,
    test_start_date='2026-08-17',
    assignment_start_date='2026-08-05',
    width=1500,
    height=400
)

metric_name = 'gems_earned'

data_gems_earned = aggregate_segment_metric(
    data_segment_metrics, 
    metric_name=metric_name)

plot_metric(
    data_gems_earned,
    metric_name,
    show_overall_diff=False,
    show_value_cumulative=False,
    show_rel_cumulative=False,
    test_start_date='2026-08-17',
    assignment_start_date='2026-08-05',
    width=1500,
    height=400
)

data_gems_velocity = aggregate_segment_ratio_metric(
    data_segment_metrics, 
    numerator='gems_spent', 
    denominator='gems_earned', 
    ratio_name='gems_velocity')

# Ratio metrics have no cumulative form here (nor in production - energy_velocity etc. only ever
# get "Active" charts in ab_trends.ipynb's own output, never "Cumulative" ones), so the
# cumulative/overall-diff pieces are turned off rather than left to fail on the missing
# sum_energy_velocity/total_users_assigned columns those need.
plot_metric(
    data_gems_velocity,
    'gems_velocity',
    show_overall_diff=False,
    show_value_cumulative=False,
    show_rel_cumulative=False,
    test_start_date='2026-08-17',
    assignment_start_date='2026-08-05',
    width=1500,
    height=400
)